In [56]:
import json
from pathlib import Path

INPUT_PATH = Path("../../infra/json/kg_extraction/bellicum_contract_kg_normalized.json")

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    normalized_kg = json.load(f)

In [57]:
kg = normalized_kg["knowledge_graph"]

entities = kg["entities"]
relations = kg["relations"]

entities_by_id = {ent["id"]: ent for ent in entities}

# with open("../../infra/json/kg_extraction/entities.json", "w", encoding="utf-8") as f:
#     json.dump(entities, f, indent=2)

relations_by_source = {}
relations_by_target = {}

for rel in relations:
    relations_by_source.setdefault(rel["source"], []).append(rel)
    relations_by_target.setdefault(rel["target"], []).append(rel)

print("Entities:", len(entities))
print("Relations:", len(relations))
print("Entity types:")

from collections import Counter
print(Counter(ent["type"] for ent in entities))
print(Counter(rel["type"] for rel in relations))

Entities: 1330
Relations: 2164
Entity types:
Counter({'Obligation': 382, 'Clause': 345, 'DefinedTerm': 147, 'Reference': 121, 'Condition': 86, 'Right': 72, 'Value': 67, 'Prohibition': 59, 'Party': 29, 'Permission': 22})
Counter({'CONTAINS': 710, 'USES': 359, 'ASSIGNS_OBLIGATION_TO': 350, 'IS_PART_OF': 338, 'REFERENCES': 144, 'GRANTS_RIGHT_TO': 111, 'DEPENDS_ON': 110, 'DEFINES': 40, 'MODIFIES_AMENDS': 2})


In [58]:
COMPARABLE_EVENT_TYPES = {
    "Obligation", #debe hacer
    "Right", # tiene derecho
    "Permission", # puede hacer
    "Prohibition", # no puede hacer
}

# Clause        → texto
# DefinedTerm   → definición
# Party         → actor
# Condition     → cuándo aplica algo
# Reference     → referencia legal
# Value         → número / fecha / %

def get_event_actor(ent):
    props = ent.get("properties", {})

    if ent["type"] == "Obligation":
        return props.get("actor")

    if ent["type"] in ["Right", "Permission"]:
        return props.get("holder")

    if ent["type"] == "Prohibition":
        return props.get("subject")

    return None


def normalize_field(value):
    if value is None:
        return None

    if isinstance(value, str):
        return value.strip().lower()

    return value


def extract_comparable_events(entities):
    events = []

    for ent in entities:
        if ent["type"] not in COMPARABLE_EVENT_TYPES:
            continue

        props = ent.get("properties", {})

        event = {
            "event_id": ent["id"],
            "event_type": ent["type"],
            "label": ent.get("label"),
            "actor": normalize_field(get_event_actor(ent)),
            "action": normalize_field(props.get("normalized_action") or props.get("action")),
            "object": normalize_field(props.get("normalized_object") or props.get("object")),
            "condition": normalize_field(props.get("condition")),
            "deadline": normalize_field(props.get("deadline")),
            "notice_period": normalize_field(props.get("notice_period")),
            "frequency": normalize_field(props.get("frequency")),
            "scope": normalize_field(props.get("scope")),
            "exception": normalize_field(props.get("exception")),
            "consequence": normalize_field(props.get("consequence")),
            "modality": normalize_field(props.get("modality")),
            "polarity": normalize_field(props.get("polarity")),
            "value_refs": props.get("value_refs", []),
            "evidence_text": ent.get("evidence_text", []),
            "confidence": ent.get("confidence", None),
        }

        events.append(event)

    return events


events = extract_comparable_events(entities)

# print("Comparable events:", len(events))
# events[:3]

print(Counter(rel["event_type"] for rel in events))

Counter({'Obligation': 382, 'Right': 72, 'Prohibition': 59, 'Permission': 22})


In [59]:
# EVENTS_OUTPUT_PATH = Path("../../infra/json/kg_extraction/bellicum_comparable_events.json")

# with open(EVENTS_OUTPUT_PATH, "w", encoding="utf-8") as f:
#     json.dump(events, f, indent=2, ensure_ascii=False)

# print("Saved:", EVENTS_OUTPUT_PATH)

In [60]:
from collections import defaultdict

groups = defaultdict(list)

for ev in events:
    key = (
        ev["actor"],
        ev["action"],
        ev["object"]
    )
    groups[key].append(ev)

In [61]:
from itertools import combinations

candidate_pairs = []

for key, group in groups.items():
    if len(group) < 2:
        continue

    for e1, e2 in combinations(group, 2):
        candidate_pairs.append((e1, e2))

In [62]:
def compare_events(e1, e2):
    conflicts = []

    # 1. conflicto clásico
    if e1["modality"] != e2["modality"]:
        conflicts.append("modality_conflict")

    if e1["polarity"] != e2["polarity"]:
        conflicts.append("polarity_conflict")

    # 🔥 2. NUEVO: obligation vs no obligation
    if e1["modality"] == "obligation" and e2["modality"] == "obligation":

        text1 = str(e1.get("evidence_text", "")).lower()
        text2 = str(e2.get("evidence_text", "")).lower()

        neg_patterns = [
            "no obligation",
            "not obligated",
            "shall not be required",
            "no requirement",
            "shall not be obligated",
        ]

        def is_negative(text):
            return any(p in text for p in neg_patterns)

        if is_negative(text1) != is_negative(text2):
            conflicts.append("obligation_vs_no_obligation")

    # 3. conflictos existentes
    if e1["condition"] != e2["condition"]:
        conflicts.append("condition_mismatch")

    if e1["scope"] != e2["scope"]:
        conflicts.append("scope_mismatch")

    return conflicts

In [63]:
results = []

for e1, e2 in candidate_pairs:
    conflicts = compare_events(e1, e2)

    if conflicts:
        results.append({
            "event_1_id": e1["event_id"],
            "event_2_id": e2["event_id"],

            "actor": e1["actor"],
            "action": e1["action"],
            "object": e1["object"],

            "event_1_modality": e1["modality"],
            "event_2_modality": e2["modality"],

            "event_1_polarity": e1["polarity"],
            "event_2_polarity": e2["polarity"],

            "event_1_text": e1.get("evidence_text"),
            "event_2_text": e2.get("evidence_text"),

            "conflicts": conflicts
        })

In [64]:
for r in results:
    print("\n" + "="*80)
    print("EVENT 1:", r["event_1_id"])
    print("MOD:", r["event_1_modality"], "| POL:", r["event_1_polarity"])
    print("TEXT:", r["event_1_text"])

    print("\nEVENT 2:", r["event_2_id"])
    print("MOD:", r["event_2_modality"], "| POL:", r["event_2_polarity"])
    print("TEXT:", r["event_2_text"])

    print("\nCONFLICTS:", r["conflicts"])


EVENT 1: obligation_bellicum_use_miltenyi_products
MOD: obligation | POL: positive
TEXT: ['Bellicum desires to use certain Miltenyi Products (as defined below) solely for the Permitted Use (as defined below) in connection with the development and manufacture of certain Bellicum Products (as defined below) by Bellicum and/or its Subcontractors or Licensees (as defined below) for use in preclinical and clinical development programs and, if approved, for commercial use', 'Bellicum shall use ... Miltenyi Products in accordance with all Applicable Laws and all requirements of Regulatory Authorities applicable to such use']

EVENT 2: obligation_use_caution_bellicum
MOD: obligation | POL: positive
TEXT: ['Bellicum acknowledges that the Miltenyi Products should be used with the same caution applied to any potentially hazardous compound.']

CONFLICTS: ['condition_mismatch', 'scope_mismatch']

EVENT 1: obligation_bellicum_use_miltenyi_products
MOD: obligation | POL: positive
TEXT: ['Bellicum de